In [6]:
import json
import requests
from urllib.parse import quote
import pandas as pd
import os

In [7]:
config_path = os.path.join(os.getcwd(), "..", "api", "api.json")


with open(config_path, "r") as f:
    config = json.load(f)

endpoint = config["endpoint"]
headers = config["headers"]

**When dowloading the whole dataset of 14 million rows it is important to not use the API, as the dowload will get very slow at a certain time. Do it via the website. The API implementation is kept if we decide to use it for further filtering of the data. After dowloading the csv make sure to either run the cell below or create a directory called data and paste the csv.**

The reason why everyone has to download it themselves is that the file is simply to large and can't be pushed.

In [8]:
# ---------------------------------------------------
# CSV PATH SETUP
# ---------------------------------------------------

data_dir = os.path.join(os.getcwd(), "..", "data")
os.makedirs(data_dir, exist_ok=True)

csv_path = os.path.join(data_dir, "chicago_mobility_data.csv")

first_batch = True

In [9]:
# ---------------------------------------------------
# CHECK IF FILE ALREADY EXISTS
# ---------------------------------------------------

if os.path.exists(csv_path):
    print("Already downloaded")
else:
    print("Starting download...")
# ---------------------------------------------------
# QUERY SETUP
# ---------------------------------------------------

    select_fields = ", ".join(config["query"]["SELECT"])

    order_fields = config["query"]["ORDER_BY"]["fields"]
    order_direction = config["query"]["ORDER_BY"]["direction"]

    order_by = f"{order_fields[0]} {order_direction}, {order_fields[1]} {order_direction}"

    limit = 100000

    current_timestamp = None
    current_id = None
    total_rows = 0

    # ---------------------------------------------------
    # LOOP
    # ---------------------------------------------------

    while True:

        if current_timestamp and current_id:
            where_clause = f"""
            WHERE (
                trip_start_timestamp < "{current_timestamp}"
                OR (
                    trip_start_timestamp = "{current_timestamp}"
                    AND trip_id < "{current_id}"
                )
            )
            """
        else:
            where_clause = ""

        sql_query = f"""
        SELECT {select_fields}
        {where_clause}
        ORDER BY {order_by}
        LIMIT {limit}
        """

        url = f"{endpoint}?query={quote(sql_query)}"
        response = requests.get(url, headers=headers)

        if response.status_code != 200:
            print(f"Error: {response.status_code} - {response.text}")
            break

        batch = response.json()

        if not batch:
            break

        df = pd.DataFrame(batch)

        df.to_csv(
            csv_path,
            mode="a",
            header=first_batch,
            index=False
        )

        first_batch = False

        current_timestamp = batch[-1]["trip_start_timestamp"]
        current_id = batch[-1]["trip_id"]

        total_rows += len(batch)
        print(
        f"Total: {total_rows:,} | "
        f"Last timestamp: {current_timestamp}"
        )   

Already downloaded


In [10]:
df = pd.read_csv(csv_path)
df.shape

(14802849, 23)

In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 14802849 entries, 0 to 14802848
Data columns (total 23 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   Trip ID                     str    
 1   Taxi ID                     str    
 2   Trip Start Timestamp        str    
 3   Trip End Timestamp          str    
 4   Trip Seconds                float64
 5   Trip Miles                  str    
 6   Pickup Census Tract         float64
 7   Dropoff Census Tract        float64
 8   Pickup Community Area       float64
 9   Dropoff Community Area      float64
 10  Fare                        str    
 11  Tips                        str    
 12  Tolls                       str    
 13  Extras                      str    
 14  Trip Total                  str    
 15  Payment Type                str    
 16  Company                     str    
 17  Pickup Centroid Latitude    str    
 18  Pickup Centroid Longitude   str    
 19  Pickup Centroid Location    st

In [12]:
df.head()

,Trip ID,Taxi ID,Trip Start Timestamp,Trip End Timestamp,Trip Seconds,Trip Miles,Pickup Census Tract,Dropoff Census Tract,Pickup Community Area,Dropoff Community Area,...,Extras,Trip Total,Payment Type,Company,Pickup Centroid Latitude,Pickup Centroid Longitude,Pickup Centroid Location,Dropoff Centroid Latitude,Dropoff Centroid Longitude,Dropoff Centroid Location
0,fcea310f20803276dda9145042f290c0c65109f6,c19d802859a6a7fa0822f0b7156d8bbebe4830062f1832...,04/01/2026 12:00:00 AM,04/01/2026 12:30:00 AM,2.280,17,NaN,NaN,76.0,28.0,...,"$6,00","$64,30",Credit Card,Transit Administrative Center Inc,"41,980264315","-87,913624596",POINT (-87.913624596 41.9802643146),"41,874005383","-87,66351755",POINT (-87.6635175498 41.874005383)
1,134b1d8f7979159300de8919d5ee0a6181e5e78a,eead1e2c21b26570411277cb384d3d8d072fc6da913b06...,04/01/2026 12:00:00 AM,04/01/2026 12:15:00 AM,1.406,"16,03",NaN,NaN,76.0,8.0,...,"$5,00","$54,90",Credit Card,5 Star Taxi,"41,980264315","-87,913624596",POINT (-87.913624596 41.9802643146),"41,899602111","-87,633308037",POINT (-87.6333080367 41.899602111)
2,15d08e4e3eaf62b6010e34507333111016b4bd13,8ff96b8befe47605908893fbb12cc9a4845f2b6f9bfc38...,04/01/2026 12:00:00 AM,04/01/2026 12:00:00 AM,0.000,0,NaN,NaN,28.0,28.0,...,"$0,00","$30,00",Credit Card,Transit Administrative Center Inc,"41,874005383","-87,66351755",POINT (-87.6635175498 41.874005383),"41,874005383","-87,66351755",POINT (-87.6635175498 41.874005383)
3,196c3c094b45b0cd2da4ae4d52ec76e3189296e9,58a80cdb3c9c1a43c90bb9b1d4faa5da222f6b7cf92bc5...,04/01/2026 12:00:00 AM,04/01/2026 12:15:00 AM,890.000,"7,71",NaN,NaN,32.0,3.0,...,"$0,00","$26,88",Credit Card,Flash Cab,"41,878865584","-87,625192142",POINT (-87.6251921424 41.8788655841),"41,96581197","-87,655878786",POINT (-87.6558787862 41.96581197)
4,19cdbf313104c8042fe1dea4d2ae3d49dff8780d,b56d49bc42507f35096478dd9613ed6aa7978739e34a99...,04/01/2026 12:00:00 AM,04/01/2026 12:15:00 AM,1.598,"17,42",1.703198e+10,1.703108e+10,76.0,8.0,...,"$4,00","$57,90",Credit Card,City Service,"41,97907082","-87,903039661",POINT (-87.9030396611 41.9790708201),"41,907520075","-87,6266589",POINT (-87.6266589003 41.9075200747)
